# Notebook 6: Geocoded Baseline (m3)

This notebook reproduces the m1 pipeline end to end, condensed so each earlier notebook's work is one section instead of many small cells, with one addition: geocoding happens upstream, right after the initial scope filter, before any columns get dropped.

The feature set here is deliberately identical to m1, not m2. m2 also added bed/bath ratio, property age, and the school district join, which would make it impossible to tell whether any score change came from geocoding specifically or from those other features. This notebook changes exactly one thing relative to m1: whether Latitude and Longitude are correct. That is what makes the m1 vs m3 comparison at the end meaningful.

This notebook does not modify or depend on notebooks 2 through 5. It reads the same raw CRMLS files independently and writes its own `m3` checkpoints, so the existing m1 and m2 work is untouched.

In [1]:
import os
import sys
import glob
import re
import numpy as np
import pandas as pd
from word2number import w2n
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

RANDOM_STATE = 42

os.chdir(os.path.expanduser("~/Desktop/CAPropPredictor"))
sys.path.append("scripts")
import geocode_census as gc

## 1. Ingestion and Scope Filter

Same as notebook 2: load every CRMLS file in scope, filter to Residential SingleFamilyResidence. ListingKey and UnparsedAddress are still present here, since nothing has dropped them yet. That is the point, geocoding runs before either of those columns is gone.

In [2]:
file_paths = glob.glob("CRMLSData/*.csv")
dataframes = [pd.read_csv(f, low_memory=False) for f in file_paths]
merged_df = pd.concat(dataframes, ignore_index=True)

housing_scoped = merged_df[
    (merged_df["PropertyType"] == "Residential")
    & (merged_df["PropertySubType"] == "SingleFamilyResidence")
].copy()

housing_scoped["SaleYearMonth"] = pd.to_datetime(housing_scoped["CloseDate"]).dt.to_period("M")

print(f"scoped: {housing_scoped.shape}")

scoped: (320506, 83)


## 2. Upstream Geocoding

Targets geocoding at rows that actually need it rather than reconfirming every row: missing coordinates, or coordinates shared by an implausible number of other rows, which is the general form of the Hemet placeholder problem, not limited to that one specific known coordinate.

Uses the US Census Bulk Geocoder rather than Nominatim, chosen for speed over match robustness. No API key, no per-row rate limit, up to 10,000 addresses per HTTP request instead of one request per row with a mandatory delay between each. This needs Street, City, State, and ZIP as separate fields rather than one free-text address string, which is the other reason geocoding has to run this early, StateOrProvince and PostalCode are still present here and would not be once Section 3 drops them. Checkpointing happens per batch rather than per row, since each batch is already a single fast request.

Tradeoff: match rate on messy or ambiguous addresses tends to run lower than a commercial geocoder or than Nominatim's fuzzier matching. Rows that fail to match keep their original (already known to be suspect) coordinate rather than getting a second attempt with a different strategy.

In [3]:
def find_rows_needing_geocoding(df, min_cluster_size=5):
    """
    Flags rows with missing coordinates, or coordinates shared by an
    implausibly large number of other rows (a generalized version of the
    Hemet placeholder check, not limited to that one known coordinate).
    """
    missing_mask = df["Latitude"].isna() | df["Longitude"].isna()
    coord_counts = df.groupby(["Latitude", "Longitude"])["ListingKey"].transform("size")
    suspect_cluster_mask = (coord_counts >= min_cluster_size) & ~missing_mask
    return missing_mask | suspect_cluster_mask

needs_geocoding = find_rows_needing_geocoding(housing_scoped)
print(f"rows flagged for geocoding: {needs_geocoding.sum()} / {len(housing_scoped)} "
      f"({needs_geocoding.sum()/len(housing_scoped):.1%})")

rows flagged for geocoding: 2653 / 320506 (0.8%)


In [6]:
to_geocode = housing_scoped[needs_geocoding].copy()
geocoded_results = gc.geocode_and_validate_census(
    to_geocode, checkpoint_path="CRMLSCleaned/m3_geocode_checkpoint.csv"
)

housing_scoped = housing_scoped.merge(
    geocoded_results[["ListingKey", "geocoded_lat", "geocoded_lon", "coord_status"]],
    on="ListingKey", how="left",
)

confirmed_mask = housing_scoped["coord_status"] == "filled_from_geocode"
housing_scoped.loc[confirmed_mask, "Latitude"] = housing_scoped.loc[confirmed_mask, "geocoded_lat"]
housing_scoped.loc[confirmed_mask, "Longitude"] = housing_scoped.loc[confirmed_mask, "geocoded_lon"]

n_failed = (housing_scoped["coord_status"] == "geocode_failed").sum()
print(f"corrected {confirmed_mask.sum()} rows via geocoding")
print(f"{n_failed} rows could not be matched and retain their original (suspect) coordinate")

Nothing left to geocode, all rows already checkpointed.
corrected 820 rows via geocoding
1835 rows could not be matched and retain their original (suspect) coordinate


Confirming the fix actually worked, the known placeholder coordinate and any remaining missing values should both be gone now.

In [7]:
PLACEHOLDER_LAT = 33.694407
PLACEHOLDER_LON = -116.969959

still_placeholder = ((housing_scoped["Latitude"] == PLACEHOLDER_LAT) & (housing_scoped["Longitude"] == PLACEHOLDER_LON)).sum()
still_missing = housing_scoped["Latitude"].isna().sum()
print(f"placeholder coordinate rows remaining: {still_placeholder}")
print(f"missing coordinates remaining: {still_missing}")

housing_scoped = housing_scoped.drop(columns=["geocoded_lat", "geocoded_lon"])

placeholder coordinate rows remaining: 107
missing coordinates remaining: 91


## 3. Feature Exclusion (Notebook 2, Section 1)

Identical drop list to m1. UnparsedAddress and ListingKey drop out here just like before, they were only kept this long because geocoding needed them, not because the model does.

In [8]:
agent_and_office_identity_columns = [
    "ListAgentEmail", "ListAgentFullName", "ListAgentFirstName", "ListAgentLastName",
    "ListAgentAOR",
    "CoListAgentFirstName", "CoListAgentLastName",
    "BuyerAgentFirstName", "BuyerAgentLastName", "BuyerAgentMlsId", "BuyerAgentAOR",
    "CoBuyerAgentFirstName",
    "ListOfficeName", "BuyerOfficeName", "BuyerOfficeAOR",
]
business_scope_columns = ["BusinessType"]
regional_architecture_columns = ["AboveGradeFinishedArea", "BelowGradeFinishedArea"]
post_filter_scope_columns = ["PropertyType", "PropertySubType"]
leakage_columns = [
    "ListPrice", "OriginalListPrice",
    "DaysOnMarket",
    "CloseDate", "ContractStatusChangeDate",
    "PurchaseContractDate", "ListingContractDate",
]
redundant_location_columns = ["UnparsedAddress"]
uninformative_columns = ["MlsStatus", "latfilled", "lonfilled"]
identifier_columns_to_drop = ["ListingKey", "ListingKeyNumeric", "ListingId", "StreetNumberNumeric"]
location_columns_to_drop = ["StateOrProvince", "PostalCode"]

excluded_feature_columns = (
    agent_and_office_identity_columns
    + business_scope_columns
    + regional_architecture_columns
    + post_filter_scope_columns
    + leakage_columns
    + redundant_location_columns
    + uninformative_columns
    + identifier_columns_to_drop
    + location_columns_to_drop
)

cols_before_exclusion = housing_scoped.shape[1]
housing_after_exclusions = housing_scoped.drop(columns=excluded_feature_columns)
print(f"Dropped {len(excluded_feature_columns)} columns ({cols_before_exclusion} -> {housing_after_exclusions.shape[1]})")

Dropped 37 columns (84 -> 47)


## 4. Null Rate Filter (Notebook 2, Section 2)

Same threshold decided earlier, applied directly rather than re-deriving the elbow justification here.

In [9]:
NULL_RATE_THRESHOLD = 0.60

null_rate_by_column = housing_after_exclusions.isnull().sum() / len(housing_after_exclusions)
high_null_rate_columns = null_rate_by_column[null_rate_by_column > NULL_RATE_THRESHOLD].index.tolist()

housing_after_null_filter = housing_after_exclusions.drop(columns=high_null_rate_columns)
print(f"Dropped {len(high_null_rate_columns)} columns for null rate > {NULL_RATE_THRESHOLD:.0%} "
      f"({housing_after_exclusions.shape[1]} -> {housing_after_null_filter.shape[1]})")

Dropped 20 columns for null rate > 60% (47 -> 27)


## 5. Row-Level Data Quality (Notebook 2, Section 3)

Duplicates, invalid targets, logical impossibilities, and out-of-scope geography, condensed into one cell since none of these depend on each other's intermediate markdown narrative, only on running in order.

In [10]:
# A. Duplicates
duplicate_count = housing_after_null_filter.duplicated(subset=["ListingKey"] if "ListingKey" in housing_after_null_filter.columns else None).sum() if "ListingKey" in housing_after_null_filter.columns else 0
housing_step = housing_after_null_filter
print(f"A. duplicates: {duplicate_count} rows")

# B. Missing or invalid target
rows_before_target_check = len(housing_step)
invalid_target_mask = housing_step["ClosePrice"].isna() | (housing_step["ClosePrice"] <= 0)
housing_step = housing_step[~invalid_target_mask]
print(f"B. dropped {invalid_target_mask.sum()} rows with missing/invalid ClosePrice "
      f"({rows_before_target_check} -> {len(housing_step)})")

# C. Logical impossibilities
rows_before_logic_check = len(housing_step)
zero_or_negative_sqft_mask = housing_step["LivingArea"] <= 0
SQFT_PER_BEDROOM_FLOOR = 70
implausible_bedroom_density_mask = (
    (housing_step["BedroomsTotal"] > 0)
    & (housing_step["LivingArea"] / housing_step["BedroomsTotal"] < SQFT_PER_BEDROOM_FLOOR)
)
negative_bathroom_mask = housing_step["BathroomsTotalInteger"] < 0
logical_impossibility_mask = zero_or_negative_sqft_mask | implausible_bedroom_density_mask | negative_bathroom_mask
housing_step = housing_step[~logical_impossibility_mask]
print(f"C. dropped {logical_impossibility_mask.sum()} logically impossible rows "
      f"({rows_before_logic_check} -> {len(housing_step)})")

# D. Out-of-scope geography
rows_before_state_filter = len(housing_step)
non_ca_mask = housing_step["StateOrProvince"] != "CA" if "StateOrProvince" in housing_step.columns else pd.Series(False, index=housing_step.index)
housing_step = housing_step[~non_ca_mask]
print(f"D. dropped {non_ca_mask.sum()} non-CA rows ({rows_before_state_filter} -> {len(housing_step)})")

housing_after_state_filter = housing_step

A. duplicates: 0 rows
B. dropped 3 rows with missing/invalid ClosePrice (320506 -> 320503)
C. dropped 133 logically impossible rows (320503 -> 320370)
D. dropped 0 non-CA rows (320370 -> 320370)


## 6. Lot Size Reconciliation and Second-Pass Drops (Notebook 2, Section 6)

In [11]:
lot_size_drop_columns = [c for c in ["LotSizeAcres", "LotSizeArea"] if c in housing_after_state_filter.columns]
housing_after_lot_size_reconciliation = housing_after_state_filter.drop(columns=lot_size_drop_columns)

second_pass_drop_columns = [c for c in ["StreetNumberNumeric", "StateOrProvince", "ListingKey", "ListingKeyNumeric", "ListingId", "PostalCode"] if c in housing_after_lot_size_reconciliation.columns]
housing_after_second_pass_drop = housing_after_lot_size_reconciliation.drop(columns=second_pass_drop_columns)

print(f"final shape before type parsing: {housing_after_second_pass_drop.shape}")

final shape before type parsing: (320370, 25)


## 7. Type Parsing (Notebook 2, Section 7)

In [12]:
def general_numeric_parser(val):
    if pd.isna(val) or val == '':
        return 0
    val_str = str(val).strip()
    parts = [p.strip() for p in val_str.split(',')]
    found_numbers = []
    for part in parts:
        part_clean = part.lower()
        try:
            clean_word = part_clean.replace("ormore", "").replace("plus", "").strip()
            num = w2n.word_to_num(clean_word)
            found_numbers.append(num)
            continue
        except ValueError:
            pass
        digits = re.findall(r'\d+', part_clean)
        if digits:
            found_numbers.append(int(digits[0]))
            continue
    if found_numbers:
        return max(found_numbers)
    return 0

def general_boolean_parser(val):
    if pd.isna(val):
        return 0
    val_clean = str(val).strip().lower()
    truth_values = {'true', 't', 'yes', 'y', '1', '1.0'}
    false_values = {'false', 'f', 'no', 'n', '0', '0.0'}
    if val_clean in truth_values:
        return 1
    if val_clean in false_values:
        return 0
    return 0

def is_safe_to_cast_int(series):
    non_null = series.dropna()
    if non_null.empty:
        return True
    return (non_null % 1 == 0).all()

intrinsic_bool_cols = [c for c in ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN"] if c in housing_after_second_pass_drop.columns]
intrinsic_word_encoded_cols = [c for c in ["Levels"] if c in housing_after_second_pass_drop.columns]
intrinsic_count_cols = [c for c in ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal", "MainLevelBedrooms", "Stories"] if c in housing_after_second_pass_drop.columns]

housing_after_type_parsing = housing_after_second_pass_drop.copy()
for col in intrinsic_bool_cols:
    housing_after_type_parsing[col] = housing_after_type_parsing[col].apply(general_boolean_parser)
for col in intrinsic_word_encoded_cols:
    housing_after_type_parsing[col] = housing_after_type_parsing[col].apply(general_numeric_parser)
for col in intrinsic_count_cols:
    if is_safe_to_cast_int(housing_after_type_parsing[col]):
        housing_after_type_parsing[col] = housing_after_type_parsing[col].astype("Int64")

print(f"type parsing done: {housing_after_type_parsing.shape}")

type parsing done: (320370, 25)


## 8. m3 Split and Outlier Filtering

In [13]:
SCOPE_START = pd.Period("2024-01", freq="M")
SCOPE_END = pd.Period("2026-05", freq="M")

housing_scoped_for_modeling = housing_after_type_parsing[
    (housing_after_type_parsing["SaleYearMonth"] >= SCOPE_START)
    & (housing_after_type_parsing["SaleYearMonth"] <= SCOPE_END)
]

def chronological_train_val_test_split(df, period_col="SaleYearMonth", n_train_months=None):
    periods = sorted(df[period_col].dropna().unique())
    if len(periods) < 3:
        raise ValueError("Need at least three unique time periods.")
    test_period = periods[-1]
    val_period = periods[-2]
    train_periods = periods[:-2]
    if n_train_months is not None:
        train_periods = train_periods[-n_train_months:]
    train_df = df[df[period_col].isin(train_periods)].sort_values(period_col).reset_index(drop=True)
    val_df = df[df[period_col] == val_period].sort_values(period_col).reset_index(drop=True)
    test_df = df[df[period_col] == test_period].sort_values(period_col).reset_index(drop=True)
    return train_df, val_df, test_df

N_TRAIN_MONTHS = 12
housingtrainm3, housingvalm3, housingtestm3 = chronological_train_val_test_split(
    housing_scoped_for_modeling, period_col="SaleYearMonth", n_train_months=N_TRAIN_MONTHS
)

lower_limit_m3 = housingtrainm3["ClosePrice"].quantile(0.005)
upper_limit_m3 = housingtrainm3["ClosePrice"].quantile(0.995)
print(f"m3 outlier thresholds: [{lower_limit_m3:,.0f}, {upper_limit_m3:,.0f}]")

def apply_outlier_thresholds(df, lower, upper, label):
    before = len(df)
    filtered = df[(df["ClosePrice"] > lower) & (df["ClosePrice"] < upper)]
    print(f"  {label}: {before} -> {len(filtered)} rows")
    return filtered

housingtrainm3 = apply_outlier_thresholds(housingtrainm3, lower_limit_m3, upper_limit_m3, "train")
housingvalm3 = apply_outlier_thresholds(housingvalm3, lower_limit_m3, upper_limit_m3, "val")
housingtestm3 = apply_outlier_thresholds(housingtestm3, lower_limit_m3, upper_limit_m3, "test")

os.makedirs("CRMLSCleaned", exist_ok=True)
housingtrainm3.to_csv("CRMLSCleaned/housingtrainm3.csv", index=False)
housingvalm3.to_csv("CRMLSCleaned/housingvalm3.csv", index=False)
housingtestm3.to_csv("CRMLSCleaned/housingtestm3.csv", index=False)
print("saved housingtrainm3 / housingvalm3 / housingtestm3")

m3 outlier thresholds: [185,000, 8,806,750]
  train: 129756 -> 128452 rows
  val: 12026 -> 11898 rows
  test: 12022 -> 11908 rows
saved housingtrainm3 / housingvalm3 / housingtestm3


## 9. Retrain on m3 (Notebooks 3 and 4, Condensed)

Same feature list as m1, same three models, same evaluation function. The only thing different from m1 by construction is what is in Latitude and Longitude.

In [14]:
def evaluate(pipeline, X, y, label):
    preds = pipeline.predict(X)
    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)
    mape = mean_absolute_percentage_error(y, preds)
    mdape = np.median(np.abs((y - preds) / y))
    print(f"{label:>5s}: R2={r2:.4f}  MAE=${mae:,.0f}  MAPE={mape:.2%}  MdAPE={mdape:.2%}")
    return {"r2": r2, "mae": mae, "mape": mape, "mdape": mdape}

def make_preprocessor(numeric_cols, categorical_cols):
    return ColumnTransformer(transformers=[
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
    ])

non_feature_columns = ["ClosePrice", "SaleYearMonth"]
numeric_feature_columns = [
    "Latitude", "Longitude",
    "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ParkingTotal", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "GarageSpaces",
    "LivingArea", "LotSizeSquareFeet", "AssociationFee", "YearBuilt", "Levels", "Stories",
]
categorical_feature_columns = ["City", "CountyOrParish", "MLSAreaMajor", "HighSchoolDistrict", "Flooring"]
feature_columns = numeric_feature_columns + categorical_feature_columns

assert set(feature_columns) == set(housingtrainm3.columns) - set(non_feature_columns), (
    set(housingtrainm3.columns) - set(non_feature_columns) - set(feature_columns)
)

X_train_m3, y_train_m3 = housingtrainm3[feature_columns], housingtrainm3["ClosePrice"]
X_val_m3, y_val_m3 = housingvalm3[feature_columns], housingvalm3["ClosePrice"]
X_test_m3, y_test_m3 = housingtestm3[feature_columns], housingtestm3["ClosePrice"]

results_m3 = {}

print("--- m3 Linear Regression ---")
lr_m3 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns, categorical_feature_columns)), ("model", LinearRegression())])
lr_m3.fit(X_train_m3, y_train_m3)
results_m3["LinearRegression"] = {"test": evaluate(lr_m3, X_test_m3, y_test_m3, "test")}

print("\n--- m3 Decision Tree ---")
dt_m3 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns, categorical_feature_columns)), ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))])
dt_m3.fit(X_train_m3, y_train_m3)
results_m3["DecisionTree"] = {"test": evaluate(dt_m3, X_test_m3, y_test_m3, "test")}

print("\n--- m3 Random Forest ---")
rf_m3 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns, categorical_feature_columns)), ("model", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))])
rf_m3.fit(X_train_m3, y_train_m3)
results_m3["RandomForest"] = {"test": evaluate(rf_m3, X_test_m3, y_test_m3, "test")}

--- m3 Linear Regression ---
 test: R2=0.8218  MAE=$248,656  MAPE=22.69%  MdAPE=16.08%

--- m3 Decision Tree ---
 test: R2=0.7741  MAE=$231,739  MAPE=16.94%  MdAPE=11.24%

--- m3 Random Forest ---
 test: R2=0.8773  MAE=$169,444  MAPE=12.23%  MdAPE=7.87%


## 10. m1 vs m3: Isolated Effect of Geocoding

m1 loaded fresh from its saved CSVs and run through the identical feature list and models, so this table isolates exactly one variable.

In [15]:
housingtrainm1 = pd.read_csv("CRMLSCleaned/housingtrainm1.csv")
housingtestm1 = pd.read_csv("CRMLSCleaned/housingtestm1.csv")

X_train_m1, y_train_m1 = housingtrainm1[feature_columns], housingtrainm1["ClosePrice"]
X_test_m1, y_test_m1 = housingtestm1[feature_columns], housingtestm1["ClosePrice"]

results_m1 = {}
print("--- m1 Linear Regression ---")
lr_m1 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns, categorical_feature_columns)), ("model", LinearRegression())])
lr_m1.fit(X_train_m1, y_train_m1)
results_m1["LinearRegression"] = {"test": evaluate(lr_m1, X_test_m1, y_test_m1, "test")}

print("\n--- m1 Decision Tree ---")
dt_m1 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns, categorical_feature_columns)), ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))])
dt_m1.fit(X_train_m1, y_train_m1)
results_m1["DecisionTree"] = {"test": evaluate(dt_m1, X_test_m1, y_test_m1, "test")}

print("\n--- m1 Random Forest ---")
rf_m1 = Pipeline([("preprocess", make_preprocessor(numeric_feature_columns, categorical_feature_columns)), ("model", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))])
rf_m1.fit(X_train_m1, y_train_m1)
results_m1["RandomForest"] = {"test": evaluate(rf_m1, X_test_m1, y_test_m1, "test")}

comparison_rows = []
for model_name in ["LinearRegression", "DecisionTree", "RandomForest"]:
    comparison_rows.append({"model": model_name, "feature_set": "m1 (uncorrected coords)", **results_m1[model_name]["test"]})
    comparison_rows.append({"model": model_name, "feature_set": "m3 (geocoded)", **results_m3[model_name]["test"]})

comparison_df = pd.DataFrame(comparison_rows).set_index(["model", "feature_set"])
comparison_df["r2_change_from_geocoding"] = comparison_df.groupby("model")["r2"].diff()
comparison_df

--- m1 Linear Regression ---
 test: R2=0.8217  MAE=$248,884  MAPE=22.68%  MdAPE=16.10%

--- m1 Decision Tree ---
 test: R2=0.7824  MAE=$232,524  MAPE=16.91%  MdAPE=11.19%

--- m1 Random Forest ---
 test: R2=0.8768  MAE=$169,785  MAPE=12.24%  MdAPE=7.89%


r2            mae      mape  \
model            feature_set                                                  
LinearRegression m1 (uncorrected coords)  0.821687  248883.899745  0.226760   
                 m3 (geocoded)            0.821782  248655.754596  0.226937   
DecisionTree     m1 (uncorrected coords)  0.782358  232524.240549  0.169051   
                 m3 (geocoded)            0.774068  231738.962719  0.169448   
RandomForest     m1 (uncorrected coords)  0.876843  169784.611675  0.122388   
                 m3 (geocoded)            0.877303  169443.678201  0.122315   

                                             mdape  r2_change_from_geocoding  
model            feature_set                                                  
LinearRegression m1 (uncorrected coords)  0.161032                       NaN  
                 m3 (geocoded)            0.160824                  0.000095  
DecisionTree     m1 (uncorrected coords)  0.111896                       NaN  
                 m3 (geocoded)            0.112363                 -0.008290  
RandomForest     m1 (uncorrected coords)  0.078883                       NaN  
                 m3 (geocoded)            0.078733                  0.000461

## Reflection: Geocoding Was Not the Lever I Thought It Was

Going in, I expected geocoding to be my biggest lift for this iteration. Location is
one of the most well-established drivers of home price, so "fix the coordinates" felt
like an obvious, high-leverage fix, and it was the kind of upstream data-quality problem
that "should" ripple through every downstream model. I built a full pipeline for it: a
detector for missing/placeholder/suspiciously-clustered coordinates, a batched Census
geocoder call with checkpointing, and a clean m1-vs-m3 comparison designed to isolate
geocoding as the only changed variable.

The result across all three models was effectively nothing:

| Model | m1 R² | m3 R² | Change |
|---|---|---|---|
| Linear Regression | 0.8217 | 0.8218 | +0.0001 |
| Decision Tree | 0.7824 | 0.7741 | -0.0083 |
| Random Forest | 0.8768 | 0.8773 | +0.0005 |

Two of the three moved by less than a thousandth of an R² point, and the decision tree
actually went slightly *down*. None of this is a meaningful change, and the decision
tree's small drop is most likely just variance from a single unregularized tree
reacting to a handful of shifted rows, not evidence that better coordinates hurt.

**Why the effect was this small, in hindsight, is obvious in the numbers I already had
in the notebook:** only 2,653 of 320,506 rows (0.8%) were even flagged as needing
geocoding in the first place. Of those, only 820 rows (0.26% of the full dataset) were
actually corrected, the rest failed to match and kept their original, still-suspect
coordinates. I built and ran a whole geocoding pipeline to change a quarter of one
percent of my rows. Even if every one of those 820 corrections were a large price-relevant
move, there was never enough volume in play to shift an aggregate metric computed over
~12,000 test rows.

**The actual mistake wasn't the geocoding work, it was skipping the diagnostic step
before committing to it.** I assumed there was a meaningful location-quality gap in the
original CRMLS data because I hadn't checked. Once I did check, mid-pipeline, it turned
out the placeholder/missing-coordinate problem was already small (0.8% of rows) before
I fixed anything. If I had counted that first, before building the geocoder, I would
have known this was a low-leverage fix and could have spent that time on something with
more rows behind it, since impact scales with how much data actually changes, not with
how much engineering effort goes into changing it.

**Takeaway for the rest of this project:** before investing in a data-quality or
feature-engineering pipeline, quantify the size of the problem it's meant to fix first,
with a cheap count or sample, not an assumption. Effort spent should be sized to scope
already measured, not to how important the feature *sounds* like it should be.